In this notebook, we start by taking a look at our main dataset of skeletal malocclusion Class III in Caucasian patients, using simple statistics.

In [1]:
library(yaml)
library(dplyr)
library(psych)
library(tibble)
Sys.setlocale("LC_ALL", "en_US.UTF-8") # for the plus_minus character


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "en_US.UTF-8/en_US.UTF-8/en_US.UTF-8/C/en_US.UTF-8/C"

In [2]:
output_path <- "../outputs/data_exploration/"
# Create the output directory if it does not exist
if (!dir.exists(output_path)) {
  dir.create(output_path, recursive = TRUE)
}

In [3]:
# Load the configuration file
config <- yaml::yaml.load_file("../config.yml")

In [4]:
# Load datasets
df_cepha_measurements <- read.csv(paste0("../", config$dataset$cephalometric_measurements))
df_patient_info <- read.csv(paste0("../", config$dataset$patient_info))
df_coordinates <- read.csv(paste0("../", config$dataset$landmark_coordinates), check.names = FALSE)

In [5]:
dim(df_cepha_measurements)

[1] 655  55

In [6]:
dim(df_patient_info)

[1] 655   7

In [7]:
dim(df_coordinates)

[1] 655  25

# Population Size

In [8]:
nrow(df_patient_info)

[1] 655

# Population

In [9]:
table(df_patient_info$Population)


Iberian 
    655 

# Gender

In [10]:
table(df_patient_info$Gender)


  F   M 
315 340 

In [11]:
df_patient_info %>%
  group_by(Population, Gender) %>%
  summarize(count = n(), .groups = "drop")

Population,Gender,count
<chr>,<chr>,<int>
Iberian,F,315
Iberian,M,340


# Cephalometric Measurements

In [12]:
df_summary_measurements <- describe(df_cepha_measurements %>% select(-Patient)) %>%
  as.data.frame() %>%
  round(2) %>%
  rownames_to_column(var = "Cephalometric_Variables") %>%
  select(Cephalometric_Variables, mean, sd) %>%
  mutate(mean_sd = paste(mean, " ± ", sd)) %>%
  select(Cephalometric_Variables, mean_sd)

df_summary_measurements %>% write.csv(file = paste0(output_path, "summary_statistics_measurements.csv"), row.names = TRUE)
df_summary_measurements

Cephalometric_Variables,mean_sd
<chr>,<chr>
FH...SN..º.,10.39 ± 3.55
SNA..º.,79.49 ± 3.97
SNB..º.,83.02 ± 4.66
ANB..º.,-3.53 ± 3.58
SND..º.,80.45 ± 4.44
Y.Axis..SGn.SN...º.,66.56 ± 4.63
SN...GoGn..º.,31.6 ± 7.02
Cranio.Mx.Base.SN.Palatal.Plane..º.,8.12 ± 3.84
Occ.Plane.to.SN..º.,15.16 ± 5.23


# Selected Landmark Coordinates

In [13]:
colnames(df_coordinates %>% select(-Patient))

[1] "X_Sella"                    "Y_Sella"                   
 [3] "X_Basion"                   "Y_Basion"                  
 [5] "X_PNS"                      "Y_PNS"                     
 [7] "X_A Point"                  "Y_A Point"                 
 [9] "X_B Point"                  "Y_B Point"                 
[11] "X_Pogonion"                 "Y_Pogonion"                
[13] "X_Menton"                   "Y_Menton"                  
[15] "X_Gonion"                   "Y_Gonion"                  
[17] "X_Ramus Point"              "Y_Ramus Point"             
[19] "X_Distal Aspect of Condyle" "Y_Distal Aspect of Condyle"
[21] "X_Condylion"                "Y_Condylion"               
[23] "X_Nasion"                   "Y_Nasion"

In [14]:
landmarks_labels <- unique(gsub("^X_|^Y_", "", colnames(df_coordinates %>% select(-Patient))))
landmarks_labels

[1] "Sella"                    "Basion"                  
 [3] "PNS"                      "A Point"                 
 [5] "B Point"                  "Pogonion"                
 [7] "Menton"                   "Gonion"                  
 [9] "Ramus Point"              "Distal Aspect of Condyle"
[11] "Condylion"                "Nasion"

In [15]:
length(landmarks_labels)

[1] 12

# Treatmemt

In [16]:
table(df_patient_info$Treatment)


no-surgery    surgery 
       292        363 